In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)
from PIL import Image
import cv2
import torch
from torch.utils.data import Dataset, DataLoader,Subset
from torchvision import transforms
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import torchinfo
from tqdm.notebook import tqdm
from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, auc
from sklearn.preprocessing import label_binarize
from random import randint, randrange


In [2]:
BASE_PATH = '/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray' # Base directory
ROOT_PATH="/kaggle/working/results"

os.mkdir(ROOT_PATH)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # Computation device

WIDTH, HEIGHT = 224, 224 # Dimensions for image resizing
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225] # Mean and Standard Deviation for normalization

INV_MEAN, INV_STD = [-0.485/0.229, -0.456/0.224, -0.406/0.225], [1/0.229, 1/0.224, 1/0.225] # Inverse Mean and Standard Deviation for denormalization

PIN_MEMORY = True if torch.cuda.is_available() else False # Pin memory for faster GPU transfer
NUM_WORKERS = os.cpu_count() # Number of workers for data loading
BATCH_SIZE=12 # Number of samples per batch

EPOCHS=20 # Number of epochs for training
LEARNING_RATE=1e-5 # Learning rate for model parameter updates

NUM_CLASSES = 2
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

print(f'Device: {DEVICE}') # Print the chosen device

Device: cuda


In [3]:
# Image transform

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

In [4]:
class InvTransform(object):
    """
    Inverse the standardization applied to images.

    Args:
        inv_mean (list of float): Mean values used to denormalize images.
        inv_std (list of float): Standard deviation values used to denormalize images.
    """
    def __init__(self, inverse_mean, inverse_std):
        # Define the transformer
        self.inverse_transform = transforms.Normalize(
            mean=inverse_mean,
            std=inverse_std,
        )

    def __call__(self, image):
        """
        Denormalize the image to its original form.

        Args:
            image (torch.Tensor): The normalized image to be denormalized.

        Returns:
            image (torch.Tensor): The denormalized image.
        """
        image = self.inverse_transform(image)

        return image

inv_transform = InvTransform(INV_MEAN, INV_STD)

In [5]:
def create_loaders():
    # Load dataset
    train_dataset = datasets.ImageFolder(BASE_PATH+'/train',transform=train_transform)
    val_dataset = datasets.ImageFolder(BASE_PATH+'/val',transform=val_transform)
    test_dataset = datasets.ImageFolder(BASE_PATH+'/test',transform=val_transform)
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, 
                            batch_size=BATCH_SIZE, 
                            shuffle=True,
                            num_workers=NUM_WORKERS,
                            pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_dataset, 
                             batch_size=BATCH_SIZE, 
                             shuffle=False,
                             num_workers=NUM_WORKERS,
                             pin_memory=PIN_MEMORY)
    val_loader = DataLoader(val_dataset, 
                            batch_size=BATCH_SIZE, 
                            shuffle=False,
                            num_workers=NUM_WORKERS,
                            pin_memory=PIN_MEMORY)

    return train_loader, val_loader, test_loader
    

In [6]:
train_loader , val_loader, test_loader = create_loaders()
len(train_loader), len(val_loader) , len(test_loader)

(435, 2, 52)

In [7]:
def plot_images_and_labels(images, labels, predicted_labels=None):
    """
    Display images alongside their actual labels and, if provided, the predicted labels.
    """
    num_samples = len(images) # Number of images to display (columns)
    num_rows = 2 if predicted_labels is None else 3 # Rows for images, actual labels, and (optionally) predicted labels

    # Define the figure and axes
    fig, axes = plt.subplots(num_rows, num_samples,
                             figsize=(num_samples*5, num_samples+(num_rows*2)),
                             dpi=500,
                            facecolor='#252627')

    for i in range(num_samples):
        
        # Transform the image for visualization
        image = inv_transform(images[i]).permute(1, 2, 0).detach().cpu().numpy()
        
        # Plot image
        axes[0, i].imshow(image, cmap='gray')
        axes[0, i].set_title('Image', fontsize=20, fontweight='bold', color='white')
        axes[0, i].axis('off')

        # Actual label
        label = CLASS_NAMES[labels[i].item()]
        axes[1, i].text(
            0.5, 0.5,
            label,
            ha='center',
            va='center',
            fontsize=22,
            color='white'
        )
        axes[1, i].set_title("Actual Label", color='white',fontsize=22)
        axes[1, i].axis('off')
    

        # Plot predicted label, if available
        if predicted_labels is not None:
            pred_label = CLASS_NAMES[predicted_labels[i].item()]
            
            axes[2, i].text(
                0.5, 0.5,
                pred_label,
                ha='center',
                va='center',
                fontsize=20,
                color='white'
            )
            axes[2, i].axis('off')


    # Add overall title
    plt.suptitle('Images and labels',fontsize=30, fontweight='bold', color='white')

    version = randint(100, 999)
    plt.savefig(f"{ROOT_PATH}/images-labels-{version}.pdf",bbox_inches="tight")
    plt.savefig(f"{ROOT_PATH}/images-labels-{version}.png",dpi=600,bbox_inches="tight")

    plt.show() # Display the plot


for images, labels in train_loader:
    plot_images_and_labels(images[:8], labels[:8])
    break  

In [ ]:
def train_model(model,name):
    device = DEVICE
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    train_loss_history = []
    val_loss_history = []
    
    # Training loop
    for epoch in range(EPOCHS):
        model.train()
        running_lose = 0.0
        train_loop = tqdm(train_loader, leave=True)
        for images,labels in train_loop:
        
            images, labels = images.to(device), labels.to(device)
    
            outputs = model(images)
            loss = criterion(outputs, labels)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_lose = running_lose + loss.item()
    
        epoch_train_loss = running_lose/ len(train_loader)
        train_loss_history.append(epoch_train_loss)
    
        # -------- Validation --------
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images,labels in val_loader:
    
                images = images.to(device)
                labels = labels.to(device).long()
    
                outputs = model(images)
                loss = criterion(outputs, labels)
    
                val_loss += loss.item()
    
        epoch_val_loss = val_loss / len(val_loader)
        val_loss_history.append(epoch_val_loss)
    
        print(f"Epoch {epoch+1}/{EPOCHS} | "
              f"Train Loss: {epoch_train_loss:.8f} | "
              f"Validation Loss: {epoch_val_loss:.8f}")
    
    train_history = pd.DataFrame({'train_loss': train_loss_history})   
    val_history = pd.DataFrame({'val_loss': val_loss_history})
    
    # Save model
    torch.save(model.state_dict(), f"{name}_classifier_chest_xray.pth")

    return model,train_history, val_history

In [8]:
# Model: Pretrained ResNet18
model = models.resnet18(pretrained=True)
name="resnet18"

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Train the model
# fine_tuned_model,tarin_history, val_history = train_model(model,name)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 143MB/s] 


In [9]:
 # load the model
MODEL_PATH ="/kaggle/input/models/atenazare/finetuned-resnet18-chestxray/pytorch/default/1/resnet18_classifier_chest_xray.pth"
model.load_state_dict(torch.load(MODEL_PATH))
fine_tuned_model = model.to(DEVICE)

In [ ]:
def plot_history(train_history, val_history):
    """
    Generates a plot comparing the training losses and validation losses over epochs.
    """
    epochs = range(1, len(train_history['train_loss']) + 1)

    plt.figure(figsize=(8, 5))

    plt.plot(epochs, train_history['train_loss'], label='Train Loss', marker='o')
    plt.plot(epochs, val_history['val_loss'], label='Validation Loss', marker='o')

    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')

    plt.xticks(epochs)          # Show every epoch as an integer
    plt.yscale("log")
    plt.grid(True)
    plt.legend()

    plt.savefig(f"{ROOT_PATH}/train-val-loss-function.pdf",bbox_inches="tight")
    plt.savefig(f"{ROOT_PATH}/train-val-loss-function.png",dpi=600,bbox_inches="tight")

    plt.show()

plot_history(tarin_history, val_history)

In [ ]:
def plot_confusion_matrix(model, dataloader, class_names):
    """
    Plot the confusion matrix for a multiclass classifier.
    """
    
    device = DEVICE
    model.to(device)
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            # In case you're using Inception with aux_logits=True
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        cmap="Blues",
        values_format="d",
        ax=ax,
        colorbar=False
    )

    plt.title("Confusion Matrix")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"{ROOT_PATH}/confusion-matrix.pdf",bbox_inches="tight")
    plt.savefig(f"{ROOT_PATH}/confusion-matrix.png",dpi=600,bbox_inches="tight")
    plt.show()


plot_confusion_matrix(fine_tuned_model,test_loader, CLASS_NAMES)

In [ ]:
# calculate accuracy
def claculate_accuracy():
    device = DEVICE
    fine_tuned_model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
    
            outputs = fine_tuned_model(images)          # shape: [batch_size, num_classes]
            _, predicted = torch.max(outputs, dim=1)
    
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    acc = f"Accuracy: {accuracy}"
    with open(f"{ROOT_PATH}/Accuracy-classifier.txt", 'w') as output:
            output.write(acc)
    print(f'Validation Accuracy: {accuracy:.2f}%')


claculate_accuracy()

In [ ]:
#calculate auc
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

def cal_auc(model,val_loader):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
    
            outputs = model(images)  # sigmoid is already in model
    
            probs = outputs[:, 1]

            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    # Convert to numpy
    y_scores = torch.cat(all_probs).numpy()
    y_true = torch.cat(all_labels).numpy()
    
    # Check for both classes
    if len(np.unique(y_true)) < 2:
        print("Cannot compute AUC or plot ROC — only one class in y_true.")
    else:
        # Compute AUC
        auc = roc_auc_score(y_true, y_scores)
        print(f"AUC: {auc:.4f}")
    
        # Compute ROC curve
        fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    
        # Plot ROC
        plt.figure(figsize=(6, 6))
        plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        plt.plot([0, 1], [0, 1], 'k--', label="Random")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend(loc="lower right")
        plt.grid()
        plt.savefig(f"{ROOT_PATH}/ROC.pdf",bbox_inches="tight")
        plt.savefig(f"{ROOT_PATH}/ROC.png",dpi=600,bbox_inches="tight")
        plt.show() 
        

cal_auc(fine_tuned_model,test_loader)

In [ ]:
#XAI METHODS


In [11]:
pip install grad-cam

Note: you may need to restart the kernel to use updated packages.


In [12]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [13]:
def convert_tensor_to_img(input_tensor):
    # Remove batch dimension
    img = input_tensor.squeeze(0).cpu()
    
    # Undo normalization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    img = img.permute(1, 2, 0).numpy()
    img = img * std + mean
    
    # Clip values to [0,1]
    img = np.clip(img, 0, 1)
    return img

In [17]:
def plot_Xmethods(img,grad_cam,Ig_attr,occlusion,lrp,actual_class,predicted_class,confidence):
    
    fig, axes = plt.subplots(4,3,figsize=(18, 12))
    
    
    # =================================
    # Row 1: Grad-CAM
    # =================================
    
    # Original image
    axes[0, 0].imshow(img)
    axes[0, 0].set_title("Original X-ray")
    axes[0, 0].axis("off")
    
    # Grad-CAM heatmap
    cam_plot = axes[0, 1].imshow(grad_cam[1],cmap="jet")
    axes[0, 1].set_title("Grad-CAM")
    axes[0, 1].axis("off")
    
    fig.colorbar(cam_plot,ax=axes[0, 1],fraction=0.046)
    
    # Grad-CAM overlay
    axes[0, 2].imshow(grad_cam[0])
    axes[0, 2].set_title("Grad-CAM Overlay")
    axes[0, 2].axis("off")
    
    # =================================
    # Row 2: Integrated Gradients
    # =================================
    
    # Original image
    axes[1, 0].imshow(img)
    axes[1, 0].set_title("Original X-ray")
    axes[1, 0].axis("off")
    
    # Integrated Gradients map
    ig_plot = axes[1, 1].imshow(Ig_attr[0],cmap="seismic")
    axes[1, 1].set_title("Integrated Gradients")
    axes[1, 1].axis("off")
    
    fig.colorbar(ig_plot,ax=axes[1, 1],fraction=0.046)
    
    # Integrated Gradients overlay
    axes[1, 2].imshow(Ig_attr[2])
    axes[1, 2].set_title("Integrated Gradients Overlay")
    axes[1, 2].axis("off")

    # =================================
    # Row 3: occlusion
    # =================================
    
    # Original image
    axes[2, 0].imshow(img)
    axes[2, 0].set_title("Original X-ray")
    axes[2, 0].axis("off")
    
    # Integrated Gradients map
    occlusion_plot = axes[2, 1].imshow(occlusion[0],cmap="seismic")
    axes[2, 1].set_title("occlusion")
    axes[2, 1].axis("off")
    
    fig.colorbar(occlusion_plot,ax=axes[2, 1],fraction=0.046)
    
    # Integrated Gradients overlay
    axes[2, 2].imshow(occlusion[2])
    axes[2, 2].set_title("occlusion Overlay")
    axes[2, 2].axis("off")


    # =================================
    # Row 4: lrp
    # =================================
    
    # Original image
    axes[3, 0].imshow(img)
    axes[3, 0].set_title("Original X-ray")
    axes[3, 0].axis("off")
    
    # Integrated Gradients map
    lrp_plot = axes[3, 1].imshow(lrp[0],cmap="seismic")
    axes[3, 1].set_title("lrp")
    axes[3, 1].axis("off")
    
    fig.colorbar(lrp_plot,ax=axes[3, 1],fraction=0.046)
    
    # Integrated Gradients overlay
    axes[3, 2].imshow(lrp[2])
    axes[3, 2].set_title("lrp Overlay")
    axes[3, 2].axis("off")

    
    # =================================
    # Figure title
    # =================================
    correct = actual_class == predicted_class
    title_color = "green" if correct else "red"    
    fig.suptitle(f"Actual: {actual_class} | "
        f"Predicted: {predicted_class} | "
        f"Confidence: {confidence:.2%}",
        fontsize=16,
        color=title_color)
    
    plt.tight_layout()
    
    plt.savefig(
        "xai_gradcam_ig_comparison.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [18]:
def create_cmad(input_tensor,pred, label):

    targets = [ClassifierOutputTarget(pred)]

    grayscale_cam = cam(
    input_tensor=input_tensor,
    targets=targets)

    cam_image = grayscale_cam[0]

    img = convert_tensor_to_img(input_tensor)

    visualization = show_cam_on_image(img,cam_image,use_rgb=True)
    
    return visualization,cam_image
    

In [19]:

import torch.nn.functional as F

target_layers = [model.layer4[-1]]

cam = GradCAM(model=model,target_layers=target_layers)

# for images, labels in test_loader:
     
#     images, labels = images.to(DEVICE), labels.to(DEVICE)

#     input_tensor = images[0].unsqueeze(0)

#     output = fine_tuned_model(input_tensor)

#     pred = output.argmax(dim=1).item()

#     prob = F.softmax(output, dim=1)
#     confidence = prob[0, pred].item()
    
#     actual_class = CLASS_NAMES[labels[0].item()]
#     predicted_class = CLASS_NAMES[pred]

#     visualization,cam_image = create_cmad(input_tensor,pred, labels[0])
#     img = convert_tensor_to_img(input_tensor)
        
#     plot_cmad(img,cam_image,visualization,actual_class,predicted_class,confidence)
    

In [20]:
pip install captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 9.4 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.


In [21]:
from captum.attr import visualization as viz
from captum.attr import IntegratedGradients
import torch.nn.functional as F

In [22]:
from captum.attr import Occlusion
import numpy as np
import matplotlib.pyplot as plt


def create_occlusion(input_tensor, img, model, pred):

    occlusion = Occlusion(model)

    attributions = occlusion.attribute(
        input_tensor,
        target=pred,
        strides=(3, 8, 8),
        sliding_window_shapes=(3, 20, 20),
        baselines=0
    )

    attr = attributions.squeeze().cpu().detach().numpy()

    # Combine RGB channels
    attr_raw = np.mean(attr, axis=0)

    # Visualization version
    attr_vis = np.abs(attr_raw)

    attr_vis = (
        attr_vis - attr_vis.min()
    ) / (
        attr_vis.max() - attr_vis.min() + 1e-8
    )

    # Overlay
    overlay = (
        0.5 * img +
        0.5 * plt.cm.hot(attr_vis)[..., :3]
    )

    overlay = np.clip(overlay, 0, 1)

    return attr_raw, attr_vis, overlay

In [30]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

from captum.attr import LayerLRP


def create_LRP(input_tensor, img, model, pred):

    model.eval()

    # Choose the layer to explain
    lrp = LayerLRP(
        model,
        model.layer3
    )

    # Compute LRP attribution
    attribution = lrp.attribute(
        input_tensor,
        target=pred
    )


    # Convert to numpy
    attr = (
        attribution
        .squeeze()
        .detach()
        .cpu()
        .numpy()
    )


    # Combine channels
    attr_raw = np.mean(
        attr,
        axis=0
    )


    # ResNet layer4 output is usually 7x7
    # Resize to original image size
    if attr_raw.shape != img.shape[:2]:

        attr_tensor = (
            torch.tensor(attr_raw)
            .unsqueeze(0)
            .unsqueeze(0)
        )

        attr_tensor = F.interpolate(
            attr_tensor,
            size=img.shape[:2],
            mode="bilinear",
            align_corners=False
        )

        attr_raw = (
            attr_tensor
            .squeeze()
            .numpy()
        )


    # Visualization version
    attr_vis = np.abs(attr_raw)

    attr_vis = (
        attr_vis - attr_vis.min()
    ) / (
        attr_vis.max() - attr_vis.min() + 1e-8
    )


    # Overlay
    overlay = (
        0.5 * img +
        0.5 * plt.cm.hot(attr_vis)[..., :3]
    )

    overlay = np.clip(
        overlay,
        0,
        1
    )


    return attr_raw, attr_vis, overlay

In [24]:

def create_IG(input_tensor, img, model, pred):
    
    ig = IntegratedGradients(model)

    baseline = torch.zeros_like(input_tensor)

    attributions, delta = ig.attribute(
        input_tensor,
        baselines=baseline,
        target=pred,
        method='gausslegendre',
        return_convergence_delta=True
    )

    # Remove batch dimension and convert to numpy
    attr = attributions.squeeze().cpu().detach().numpy()

    # If image has 3 channels, combine them
    attr_raw = np.mean(attr, axis=0)


    # -------------------------
    # For visualization overlay
    # -------------------------
    attr_vis = np.abs(attr_raw)

    attr_vis = (
        attr_vis - attr_vis.min()
    ) / (
        attr_vis.max() - attr_vis.min() + 1e-8
    )


    # Create overlay
    overlay = img.copy()

    overlay = (
        0.5 * img +
        0.5 * plt.cm.hot(attr_vis)[..., :3]
    )

    overlay = np.clip(overlay,0,1)


    return attr_raw, attr_vis, overlay, delta

    

In [32]:
for images, labels in test_loader:
     
    images, labels = images.to(DEVICE), labels.to(DEVICE)

    input_tensor = images[0].unsqueeze(0)

    output = fine_tuned_model(input_tensor)

    pred = output.argmax(dim=1).item()

    prob = F.softmax(output, dim=1)
    confidence = prob[0, pred].item()
    
    actual_class = CLASS_NAMES[labels[0].item()]
    predicted_class = CLASS_NAMES[pred]

    
    visualization,cam_image = create_cmad(input_tensor,pred, labels[0])
    img = convert_tensor_to_img(input_tensor)

    attr_raw, attr_vis, ig_overlay, delta = create_IG(input_tensor,img,fine_tuned_model,pred)

    occlusion_raw, occlusion_vis, occlusion_overlay = create_occlusion(input_tensor,img,fine_tuned_model,pred)

    lrp_raw, lrp_vis, lrp_overlay = create_LRP(input_tensor,img,fine_tuned_model,pred)

    grad_cam =[]
    Ig_attr=[]
    occlusion = []
    lrp=[]
    
    grad_cam.append(visualization)
    grad_cam.append(cam_image)

    Ig_attr.append(attr_raw)
    Ig_attr.append(attr_vis)
    Ig_attr.append(ig_overlay)

    occlusion.append(occlusion_raw)
    occlusion.append(occlusion_vis)
    occlusion.append(occlusion_overlay)

    lrp.append(lrp_raw)
    lrp.append(lrp_vis)
    lrp.append(lrp_overlay)

    plot_Xmethods(img,grad_cam,Ig_attr,occlusion,lrp,actual_class,predicted_class,confidence)
    
    break
    
    

AttributeError: 'Sequential' object has no attribute 'rule'

In [ ]:
import os
import shutil

folder = '/kaggle/working/results'
for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print(f'Failed to delete {file_path}. Reason: {e}')